# Linear Regression — Building Energy Efficiency Decision Model

**Goal:** estimate building heating load transparently, compare ordinary Linear Regression with sensible alternatives, visualise the data and residuals, and turn predictions into a simple design-review decision.

**Dataset:** UCI Energy Efficiency (768 building configurations, 8 design features, CC BY 4.0, DOI 10.24432/C51307).


## 1. Load the real dataset and inspect it directly
The notebook deliberately shows the analysis instead of hiding everything in helper functions.


In [ ]:
from pathlib import Path
import json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
warnings.filterwarnings('ignore')
SEED = 42
rng = np.random.default_rng(SEED)
energy = fetch_ucirepo(id=242)
X_raw = energy.data.features.copy()
y_raw = energy.data.targets.copy()
feature_names = ['relative_compactness','surface_area','wall_area','roof_area','overall_height','orientation','glazing_area','glazing_area_distribution']
target_names = ['heating_load','cooling_load']
X_raw.columns = feature_names
y_raw = y_raw.iloc[:, :2].copy()
y_raw.columns = target_names
df = pd.concat([X_raw, y_raw], axis=1)
print('Shape:', df.shape)
display(df.head())
display(df.describe().T.round(3))
audit = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique(dropna=False)})
display(audit)
print('Duplicate rows:', int(df.duplicated().sum()))


## 2. EDA and visualisation
Before modelling, inspect distributions, relationships, categories and correlation.


In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df['heating_load'], bins=28, alpha=0.8)
plt.axvline(df['heating_load'].median(), linestyle='--', label=f"median={df['heating_load'].median():.2f}")
plt.title('Heating-load distribution')
plt.xlabel('Heating Load')
plt.ylabel('Buildings')
plt.legend()
plt.tight_layout()
plt.show()
plt.figure(figsize=(7,5))
plt.scatter(df['heating_load'], df['cooling_load'], alpha=0.45, s=22)
plt.xlabel('Heating Load')
plt.ylabel('Cooling Load')
plt.title('Heating vs cooling load')
plt.tight_layout()
plt.show()
for col in ['relative_compactness','surface_area','wall_area','roof_area','overall_height','glazing_area']:
    plt.figure(figsize=(7,4))
    plt.scatter(df[col], df['heating_load'], alpha=0.40, s=20)
    plt.xlabel(col)
    plt.ylabel('Heating Load')
    plt.title(f'Heating Load vs {col}')
    plt.tight_layout()
    plt.show()
orientation_view = df.groupby('orientation', as_index=False)['heating_load'].agg(['mean','median','count']).reset_index()
display(orientation_view.round(3))
plt.figure(figsize=(7,4))
plt.bar(orientation_view['orientation'].astype(str), orientation_view['mean'])
plt.xlabel('Orientation')
plt.ylabel('Mean Heating Load')
plt.title('Heating load by orientation')
plt.tight_layout()
plt.show()
glazing_view = df.groupby('glazing_area', as_index=False)['heating_load'].agg(['mean','median','count']).reset_index()
display(glazing_view.round(3))
plt.figure(figsize=(7,4))
plt.bar(glazing_view['glazing_area'].astype(str), glazing_view['mean'])
plt.xlabel('Glazing area')
plt.ylabel('Mean Heating Load')
plt.title('Heating load by glazing area')
plt.tight_layout()
plt.show()
corr = df.corr(numeric_only=True)
plt.figure(figsize=(9,7))
image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
plt.colorbar(image, label='Correlation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=70, ha='right')
plt.yticks(range(len(corr.index)), corr.index)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()


## 3. Baseline, ordinary Linear Regression and alternatives
Orientation and glazing-distribution are treated as categories. Numeric variables are imputed/scaled inside the pipeline to keep preprocessing leakage-safe.


In [ ]:
features = feature_names
target = 'heating_load'
categorical = ['orientation','glazing_area_distribution']
numeric = [c for c in features if c not in categorical]
X = df[features].copy()
y = df[target].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('numeric', numeric_pipe, numeric), ('categorical', categorical_pipe, categorical)])
baseline = DummyRegressor(strategy='median').fit(X_train, y_train)
linear = Pipeline([('preprocess', preprocessor), ('model', LinearRegression())]).fit(X_train, y_train)
ridge = Pipeline([('preprocess', preprocessor), ('model', RidgeCV(alphas=np.logspace(-4,4,80)))]).fit(X_train, y_train)
lasso = Pipeline([('preprocess', preprocessor), ('model', LassoCV(alphas=np.logspace(-4,1,80), cv=5, random_state=SEED, max_iter=100000))]).fit(X_train, y_train)
poly_pre = ColumnTransformer([('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('poly', PolynomialFeatures(degree=2, include_bias=False)), ('scale', StandardScaler())]), numeric), ('categorical', categorical_pipe, categorical)])
poly = Pipeline([('preprocess', poly_pre), ('model', RidgeCV(alphas=np.logspace(-4,4,80)))]).fit(X_train, y_train)
models = {'dummy_median': baseline, 'linear_regression': linear, 'ridge': ridge, 'lasso': lasso, 'polynomial_ridge': poly}
rows = []
predictions = {}
for name, model in models.items():
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append({'model': name, 'mae': mean_absolute_error(y_test,pred), 'rmse': math.sqrt(mean_squared_error(y_test,pred)), 'r2': r2_score(y_test,pred)})
metrics = pd.DataFrame(rows).sort_values('rmse')
display(metrics.round(4))
plt.figure(figsize=(8,4))
plt.bar(metrics['model'], metrics['rmse'])
plt.ylabel('RMSE')
plt.title('Holdout model comparison')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
best_name = metrics.iloc[0]['model']
print('Best holdout model:', best_name)
print('Ridge alpha:', ridge.named_steps['model'].alpha_)
print('Lasso alpha:', lasso.named_steps['model'].alpha_)


## 4. Cross-validation and residual/error analysis
A strong project does not stop at one holdout score. Compare folds and inspect where the transparent linear model is wrong.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for name, model in {'linear_regression': linear, 'ridge': ridge, 'polynomial_ridge': poly}.items():
    fold_scores = -cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')
    cv_rows.append({'model': name, 'cv_rmse_mean': fold_scores.mean(), 'cv_rmse_std': fold_scores.std()})
cv_results = pd.DataFrame(cv_rows).sort_values('cv_rmse_mean')
display(cv_results.round(4))
linear_pred = predictions['linear_regression']
result = X_test.reset_index(drop=False).rename(columns={'index':'source_index'})
result['actual'] = y_test.reset_index(drop=True)
result['predicted'] = linear_pred
result['residual'] = result['actual'] - result['predicted']
result['abs_error'] = result['residual'].abs()
display(result.nlargest(12, 'abs_error').round(3))
plt.figure(figsize=(7,5))
plt.scatter(result['actual'], result['predicted'], alpha=0.60, s=25)
lo = min(result['actual'].min(), result['predicted'].min())
hi = max(result['actual'].max(), result['predicted'].max())
plt.plot([lo,hi],[lo,hi], linestyle='--')
plt.xlabel('Actual Heating Load')
plt.ylabel('Linear Regression prediction')
plt.title('Actual vs predicted')
plt.tight_layout()
plt.show()
plt.figure(figsize=(7,5))
plt.scatter(result['predicted'], result['residual'], alpha=0.60, s=25)
plt.axhline(0, linestyle='--')
plt.xlabel('Predicted')
plt.ylabel('Residual')
plt.title('Residuals vs predicted')
plt.tight_layout()
plt.show()
plt.figure(figsize=(7,4))
plt.hist(result['residual'], bins=25, alpha=0.80)
plt.axvline(0, linestyle='--')
plt.xlabel('Residual')
plt.ylabel('Rows')
plt.title('Residual distribution')
plt.tight_layout()
plt.show()
orientation_error = result.groupby('orientation', as_index=False)['abs_error'].agg(['mean','median','count']).reset_index()
display(orientation_error.round(4))
plt.figure(figsize=(7,4))
plt.bar(orientation_error['orientation'].astype(str), orientation_error['mean'])
plt.xlabel('Orientation')
plt.ylabel('MAE')
plt.title('Linear Regression error by orientation')
plt.tight_layout()
plt.show()


## 5. Coefficients, uncertainty and the decision layer
Coefficients are useful for transparency but are not causal effects. Correlated physical design variables make that distinction important.


In [ ]:
prep = linear.named_steps['preprocess']
ols = linear.named_steps['model']
cat_names = prep.named_transformers_['categorical'].named_steps['onehot'].get_feature_names_out(categorical).tolist()
names = numeric + cat_names
coef = pd.DataFrame({'feature': names, 'coefficient': ols.coef_})
coef['abs_coefficient'] = coef['coefficient'].abs()
coef = coef.sort_values('abs_coefficient', ascending=False)
display(coef.round(4))
plot_coef = coef.head(15).sort_values('coefficient')
plt.figure(figsize=(8,5))
plt.barh(plot_coef['feature'], plot_coef['coefficient'])
plt.xlabel('Coefficient')
plt.title('Linear Regression coefficients')
plt.tight_layout()
plt.show()
bootstrap_predictions = []
bootstrap_rmse = []
for i in range(150):
    positions = rng.integers(0, len(X_train), size=len(X_train))
    X_boot = X_train.iloc[positions]
    y_boot = y_train.iloc[positions]
    boot = Pipeline([('preprocess', preprocessor), ('model', LinearRegression())]).fit(X_boot, y_boot)
    pred = boot.predict(X_test)
    bootstrap_predictions.append(pred)
    bootstrap_rmse.append(math.sqrt(mean_squared_error(y_test, pred)))
bootstrap_predictions = np.vstack(bootstrap_predictions)
result['p05'] = np.quantile(bootstrap_predictions, 0.05, axis=0)
result['p50'] = np.quantile(bootstrap_predictions, 0.50, axis=0)
result['p95'] = np.quantile(bootstrap_predictions, 0.95, axis=0)
result['interval_width'] = result['p95'] - result['p05']
print('Bootstrap RMSE mean:', np.mean(bootstrap_rmse))
print('Bootstrap RMSE 5%-95%:', np.quantile(bootstrap_rmse, [0.05,0.95]))
print('Mean prediction interval width:', result['interval_width'].mean())
plt.figure(figsize=(7,4))
plt.hist(bootstrap_rmse, bins=25, alpha=0.8)
plt.xlabel('Bootstrap RMSE')
plt.ylabel('Refits')
plt.title('Bootstrap uncertainty')
plt.tight_layout()
plt.show()
review_threshold = y_train.quantile(0.75)
result['decision'] = np.where(result['predicted'] >= review_threshold, 'HIGH LOAD - REVIEW', 'LOW / NORMAL LOAD')
display(result[['actual','predicted','p05','p95','decision']].head(20).round(3))
print('Review threshold:', review_threshold)
print(result['decision'].value_counts())


## Conclusion / solution
Ordinary Linear Regression is retained as the transparent baseline and screening model. The project compares it with regularised and polynomial alternatives rather than assuming complexity is automatically better. Predictions above the training 75th-percentile heating-load threshold are flagged for design review, with bootstrap uncertainty shown alongside the estimate.

## Limitations and next steps
The UCI dataset is simulated and small. Coefficients are associations, not causal effects. A production building-energy workflow would require real measured buildings, climate/location variables, external validation, stronger uncertainty modelling and domain-engineering review.


## Reproducibility
Run `python run.py` from this project folder to recreate metrics, tables, models and PNG visualisations in `results/`. The engineering appendix added by the portfolio sync exposes the complete canonical application source inside this notebook as well.
